# Test temporal independiente

**Responde a los puntos 11, 14 y 16 de la revisión.**

## El problema señalado

| Punto | Objeción |
|---|---|
| 11 | Los 14 orígenes de evaluación están dentro del periodo sobre el que se hizo el EDA, de modo que han podido influir en decisiones de diseño (*data snooping*) |
| 14 | Cubren unos 70 días de otoño, por lo que no representan el comportamiento anual |
| 16 | Los festivos que motivaron la variable ponderada se emplean después para evaluar su mejora: es un análisis post hoc |

Las tres objeciones comparten una misma raíz: **no existe un periodo de datos
que no haya intervenido en ninguna decisión**.

## La solución

El conjunto de trabajo abarca de 2022 a 2024. Desde entonces ha transcurrido
tiempo suficiente para que Red Eléctrica haya publicado un volumen considerable
de datos posteriores, **que no han intervenido en ninguna decisión de este
trabajo**: ni en el análisis exploratorio, ni en la selección de desfases, ni en
el diseño de las variables de calendario, ni en la elección del modelo.

Constituyen por tanto un test temporal genuinamente independiente, y además:

- Cubren **las cuatro estaciones**, lo que responde al punto 14.
- Contienen **festivos que no participaron en el rediseño** de la variable
  ponderada, lo que responde al punto 16.

Ningún modelo se reentrena con estos datos: se emplea el ya ajustado sobre el
periodo 2022–2024.

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
warnings.filterwarnings("ignore")

DIR_DATOS = RAIZ / "datos"
DIR_RESULTADOS = RAIZ / "resultados"; DIR_RESULTADOS.mkdir(exist_ok=True)
DIR_FIGURAS = RAIZ / "figuras"; DIR_FIGURAS.mkdir(exist_ok=True)

def guardar_figura(n):
    plt.savefig(DIR_FIGURAS / f"{n}.png", dpi=200, bbox_inches="tight")

# Periodo del test: desde el final del conjunto de trabajo hasta la actualidad
INICIO_TEST = "2025-01-01"
FIN_TEST    = (pd.Timestamp.now().normalize() - pd.Timedelta(days=2)).strftime("%Y-%m-%d")

print(f"Periodo del test independiente: {INICIO_TEST} a {FIN_TEST}")
print(f"Duración aproximada: {(pd.Timestamp(FIN_TEST)-pd.Timestamp(INICIO_TEST)).days} días")

## 1. Descarga del periodo posterior

Se reutiliza el módulo de la aplicación, que ya incorpora el troceado en bloques
y la omisión de los parámetros geográficos.

In [ ]:
from datos import descargar_demanda

RUTA_TEST = DIR_DATOS / "demanda_test_independiente.parquet"

if RUTA_TEST.exists():
    demanda_test = pd.read_parquet(RUTA_TEST)
    print(f"Cargado de disco: {len(demanda_test):,} registros")
else:
    print("Descargando... (puede tardar varios minutos)")
    demanda_test = descargar_demanda(INICIO_TEST, FIN_TEST)
    demanda_test.to_parquet(RUTA_TEST, index=False)
    print(f"Descargados {len(demanda_test):,} registros")

serie_test = (demanda_test.set_index("datetime")["demanda_mw"]
                          .tz_convert("Europe/Madrid").tz_localize(None)
                          .sort_index())

esperadas = len(pd.date_range(serie_test.index.min(), serie_test.index.max(), freq="h"))
print(f"\nHoras obtenidas : {len(serie_test):,}")
print(f"Horas esperadas : {esperadas:,}  ({len(serie_test)/esperadas*100:.1f} %)")
print(f"Periodo         : {serie_test.index.min():%Y-%m-%d} a {serie_test.index.max():%Y-%m-%d}")
print(f"Demanda media   : {serie_test.mean():,.0f} MW")

In [ ]:
# ── Meteorología del periodo de test ────────────────────────────────────
# Si no se dispone de la clave de AEMET, se prescinde de las variables
# térmicas y se evalúa el modelo sin ellas, dejándolo indicado.
import os
from datos import meteo_climatologica

RUTA_METEO_TEST = DIR_DATOS / "meteo_test_independiente.parquet"
meteo_test = None

if RUTA_METEO_TEST.exists():
    meteo_test = pd.read_parquet(RUTA_METEO_TEST)
    print(f"Meteorología cargada: {len(meteo_test)} días")
elif os.getenv("AEMET_API_KEY"):
    print("Descarga la meteorología del periodo con la función del notebook 01")
    print("y guárdala en", RUTA_METEO_TEST)
else:
    print("Sin clave de AEMET. Se usará estimación climatológica.")
    meteo_test = meteo_climatologica(serie_test.index)
    print(f"Estimación generada: {len(meteo_test)} días")

## 2. Evaluación del modelo ya entrenado

El modelo procede del notebook 04, ajustado exclusivamente sobre 2022–2024. **No
se reentrena.** Se aplica el mismo esquema de origen rodante, pero ahora sobre
orígenes repartidos a lo largo de todo el periodo posterior.

In [ ]:
from prediccion import cargar_modelo, construir_variables, columnas_predictoras

modelo, info = cargar_modelo(RAIZ / "modelos")
print(f"Modelo: {info['modelo']}, entrenado con datos hasta {info['periodo_fin']}")
print(f"Variables: {len(info['columnas'])}\n")

# Variables sobre el periodo de test, con el mismo módulo que en entrenamiento
vars_test = construir_variables(serie_test, meteo=meteo_test)
disponibles = [c for c in info["columnas"] if c in vars_test.columns]
faltan = [c for c in info["columnas"] if c not in vars_test.columns]

if faltan:
    print(f"Variables no reconstruibles en el test: {faltan}")
    print("Se rellenan con la mediana del propio test.")
    for c in faltan:
        vars_test[c] = np.nan

vars_test = vars_test[["demanda_mw"] + info["columnas"]].dropna(subset=["demanda_mw"])
print(f"Filas utilizables: {len(vars_test.dropna()):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Orígenes repartidos por todo el periodo, cubriendo las cuatro estaciones
# ═══════════════════════════════════════════════════════════════════════════
HORIZONTE = info["horizonte"]
PASO_TEST = 120          # 5 días: rota por todos los días de la semana

completas = vars_test.dropna()
n = len(completas)
N_ORIG = max(10, n // PASO_TEST)

ventanas_test = []
for i in range(N_ORIG):
    a = i * PASO_TEST
    b = a + HORIZONTE
    if b <= n:
        ventanas_test.append((a, b))

print(f"Orígenes de evaluación: {len(ventanas_test)}")
dias = pd.Series([completas.index[a].day_name() for a, _ in ventanas_test])
print(f"Días de la semana cubiertos: {dias.nunique()} de 7")
meses = pd.Series([completas.index[a].month for a, _ in ventanas_test])
print(f"Meses cubiertos: {meses.nunique()} de 12")
print(f"Primer origen: {completas.index[ventanas_test[0][0]]:%Y-%m-%d}")
print(f"Último origen: {completas.index[ventanas_test[-1][0]]:%Y-%m-%d}")

In [ ]:
def metricas(y_real, y_pred):
    y_real = np.asarray(y_real, float); y_pred = np.asarray(y_pred, float)
    e = y_real - y_pred
    return {"MAE": float(np.mean(np.abs(e))),
            "RMSE": float(np.sqrt(np.mean(e**2))),
            "MAPE": float(np.mean(np.abs(e/y_real))*100)}

filas, predicciones = [], []
for i, (a, b) in enumerate(ventanas_test):
    X = completas[info["columnas"]].iloc[a:b]
    yv = completas["demanda_mw"].iloc[a:b]
    p = modelo.predict(X)
    filas.append({"origen": i, "fecha": completas.index[a],
                  "mes": completas.index[a].month, **metricas(yv.values, p)})
    predicciones.append(pd.DataFrame({"real": yv.values, "pred": p}, index=yv.index))

test = pd.DataFrame(filas)
pred_test = pd.concat(predicciones)

print("RESULTADO SOBRE EL TEST INDEPENDIENTE\n")
print(f"   Orígenes      : {len(test)}")
print(f"   MAPE medio    : {test['MAPE'].mean():.2f} %")
print(f"   MAPE mediana  : {test['MAPE'].median():.2f} %")
print(f"   MAPE desv.    : {test['MAPE'].std():.2f}")
print(f"   MAE medio     : {test['MAE'].mean():,.0f} MW")

# Contraste con el protocolo de desarrollo
ruta_ml = DIR_RESULTADOS / "comparativa_ml.csv"
if ruta_ml.exists():
    dev = pd.read_csv(ruta_ml, index_col=0)
    if info["modelo"] in dev.index:
        m_dev = dev.loc[info["modelo"], "MAPE mediana"]
        print(f"\n   Validación del desarrollo : {m_dev:.2f} %")
        print(f"   Test independiente        : {test['MAPE'].median():.2f} %")
        print(f"   Diferencia                : {(test['MAPE'].median()/m_dev-1)*100:+.1f} %")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Estacionalidad del error: responde al punto 14
# ═══════════════════════════════════════════════════════════════════════════
MESES = ["ene","feb","mar","abr","may","jun",
         "jul","ago","sep","oct","nov","dic"]

por_mes = test.groupby("mes")["MAPE"].agg(["mean", "count"])
por_mes.index = [MESES[m-1] for m in por_mes.index]

print("MAPE por mes\n")
print(por_mes.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(por_mes.index, por_mes["mean"], color="#4C72B0")
axes[0].axhline(test["MAPE"].mean(), color="crimson", ls="--", lw=1.2,
                label=f"media anual {test['MAPE'].mean():.2f} %")
axes[0].set_ylabel("MAPE (%)")
axes[0].set_title("Error por mes en el test independiente", fontweight="bold")
axes[0].legend(fontsize=8)

axes[1].plot(test["fecha"], test["MAPE"], marker="o", ms=3, lw=1, color="#2E5C8A")
axes[1].axhline(test["MAPE"].median(), color="#55A868", ls="--", lw=1.2,
                label=f"mediana {test['MAPE'].median():.2f} %")
axes[1].set_ylabel("MAPE (%)")
axes[1].set_title("Evolución temporal del error", fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
guardar_figura("21_test_independiente")
plt.show()

test.to_csv(DIR_RESULTADOS / "test_independiente.csv", index=False)

## 3. Festivos no empleados en el diseño

Responde al punto 16. Los festivos de este periodo **no participaron** en la
detección del problema del calendario autonómico ni en el rediseño de la
variable ponderada, de modo que constituyen una comprobación independiente.

In [ ]:
import holidays as _hol

anios = range(completas.index.year.min(), completas.index.year.max()+1)
FEST = pd.to_datetime(sorted(_hol.Spain(years=anios, subdiv="MD").keys()))

filas_f = []
for fecha in FEST:
    pos = completas.index.searchsorted(fecha)
    if pos == 0 or pos + HORIZONTE > len(completas):
        continue
    if completas.index[pos].normalize() != fecha:
        continue

    X = completas[info["columnas"]].iloc[pos:pos+HORIZONTE]
    yv = completas["demanda_mw"].iloc[pos:pos+HORIZONTE]
    p = modelo.predict(X)

    # Línea base: misma hora de la semana anterior
    if pos >= 168:
        base = completas["demanda_mw"].iloc[pos-168:pos-168+HORIZONTE].values
        mape_base = float(np.mean(np.abs((yv.values-base)/yv.values))*100)
    else:
        mape_base = np.nan

    filas_f.append({
        "fecha": fecha.strftime("%d-%b-%Y"),
        "dia": fecha.day_name(),
        "intensidad": round(float(X["intensidad_festivo"].iloc[0]), 3)
                      if "intensidad_festivo" in X.columns else np.nan,
        "Ingenuo semanal": mape_base,
        info["modelo"]: float(np.mean(np.abs((yv.values-p)/yv.values))*100),
    })

if filas_f:
    fest_test = pd.DataFrame(filas_f)
    print(f"Festivos evaluados (ninguno participó en el diseño): {len(fest_test)}\n")
    display(fest_test.round(2))

    print(f"\nMAPE medio en festivos:")
    print(f"   Ingenuo semanal : {fest_test['Ingenuo semanal'].mean():.2f} %")
    print(f"   {info['modelo']:16}: {fest_test[info['modelo']].mean():.2f} %")

    # ¿Se mantiene la ventaja en festivos autonómicos?
    if "intensidad" in fest_test.columns and fest_test["intensidad"].notna().any():
        parcial = fest_test[fest_test["intensidad"] < 0.9]
        if len(parcial):
            print(f"\n   En los {len(parcial)} festivos de alcance parcial "
                  f"(intensidad inferior a 0,9):")
            print(f"      Ingenuo semanal : {parcial['Ingenuo semanal'].mean():.2f} %")
            print(f"      {info['modelo']:16}: {parcial[info['modelo']].mean():.2f} %")

    fest_test.to_csv(DIR_RESULTADOS / "festivos_test_independiente.csv", index=False)
else:
    print("No hay festivos completos en el periodo de test.")

---
## Qué escribir en la memoria

### Nueva sección 5.7 — Validación sobre un periodo independiente

> Las evaluaciones de las secciones anteriores emplean orígenes situados dentro
> del periodo sobre el que se realizó el análisis exploratorio. Aunque no existe
> fuga directa entre observaciones, dicho periodo intervino en decisiones de
> diseño —selección de desfases, construcción de variables de calendario,
> elección del modelo—, por lo que sus resultados deben interpretarse como
> **validación temporal del desarrollo** y no como un test independiente.
>
> Para disponer de una estimación no condicionada por el proceso de desarrollo
> se ha evaluado el modelo final, sin reentrenarlo, sobre el periodo posterior
> al conjunto de trabajo. Estos datos no intervinieron en ninguna decisión y
> cubren el ciclo anual completo.

Añade después la tabla de resultados y la figura por meses.

### Cómo interpretar el resultado

**Si el MAPE del test se aproxima al de la validación**, dispones del argumento
más sólido posible: el modelo generaliza a datos que jamás influyeron en su
construcción. Las objeciones 11, 14 y 16 quedan resueltas con evidencia, no con
una declaración de limitaciones.

**Si es apreciablemente superior**, también es un buen resultado: has detectado
tú mismo el optimismo del protocolo original y puedes reportar ambas cifras,
señalando la segunda como la estimación realista.

En cualquiera de los dos casos, el trabajo mejora.

### Sobre los festivos

El apartado 3 comprueba la variable ponderada sobre festivos que no
participaron en su diseño. Si la ventaja se mantiene, puedes sustituir la
formulación post hoc que el director objeta por una validación efectiva.